In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# ---------- Utility functions ----------

def unit_vector(v, eps=1e-8):
    norm = np.linalg.norm(v, axis=-1, keepdims=True)
    return v / np.clip(norm, eps, None)


def running_mean_std(z_samples, axis=0, eps=1e-8):
    mu = np.mean(z_samples, axis=axis)
    sigma = np.std(z_samples, axis=axis) + eps
    return mu, sigma


def latin_hypercube_sampling(num_points, dim, low=-2.0, high=2.0, rng=None):
    if rng is None:
        rng = np.random.RandomState()
    cut = np.linspace(0, 1, num_points + 1)
    u = rng.rand(num_points, dim)
    a = cut[:num_points]
    b = cut[1:num_points + 1]
    rdpoints = u * (b - a)[:, None] + a[:, None]
    H = np.zeros_like(rdpoints)
    for j in range(dim):
        order = rng.permutation(num_points)
        H[:, j] = rdpoints[order, j]
    return low + (high - low) * H


# ---------- Latent manager ----------

class LatentSpaceManager:
    def __init__(self, latent_dim, init_c=2.0, rng=None):
        self.latent_dim = latent_dim
        self.init_c = init_c
        self.rng = np.random.RandomState() if rng is None else rng
        self.mu = np.zeros(latent_dim)
        self.sigma = np.ones(latent_dim)

    def update_stats(self, z_batch):
        self.mu, self.sigma = running_mean_std(z_batch)

    def to_standardized(self, z):
        return (z - self.mu) / self.sigma

    def from_standardized(self, z_tilde):
        return self.mu + self.sigma * z_tilde

    def lhs_initial_latents(self, num_agents):
        z_tilde = latin_hypercube_sampling(
            num_points=num_agents,
            dim=self.latent_dim,
            low=-self.init_c,
            high=self.init_c,
            rng=self.rng
        )
        z = self.from_standardized(z_tilde)
        return z


# ---------- Planner ----------

class LatentExplorationPlanner:
    def __init__(
        self,
        latent_dim,
        num_candidates=16,
        center_weight=0.1,
        home_weight=0.1,
        memory_weight=0.5,
        repel_weight=0.2,
        novelty_weight=1.0,
        spacing_weight=1.0,
        center_align_weight=0.1,
        memory_smooth=0.5,
        repel_radius=1.0,
        step_size=0.5,
        rng=None,
    ):
        self.d = latent_dim
        self.K = num_candidates
        self.w_c = center_weight
        self.w_h = home_weight
        self.w_m = memory_weight
        self.w_r = repel_weight
        self.lambda_nov = novelty_weight
        self.lambda_space = spacing_weight
        self.lambda_center = center_align_weight
        self.gamma = memory_smooth
        self.repel_radius = repel_radius
        self.step_size = step_size
        self.rng = np.random.RandomState() if rng is None else rng

        self.z_home = None
        self.memory = None
        self.archive = []

    def reset_agents(self, z_init):
        self.z_home = np.array(z_init, copy=True)
        self.memory = np.zeros_like(z_init)
        self.archive = [z.copy() for z in z_init]

    def _latent_center(self, z_agents):
        return np.mean(z_agents, axis=0)

    def _repulsion_direction(self, z_agents, i):
        zi = z_agents[i]
        diff = zi - z_agents
        dist = np.linalg.norm(diff, axis=1, keepdims=True)
        dist[i] = np.inf
        mask = (dist < self.repel_radius).astype(np.float32)
        safe_dist = np.clip(dist, 1e-8, None)
        contrib = mask * (diff / safe_dist)
        vec = np.sum(contrib, axis=0)
        if np.allclose(vec, 0.0):
            return np.zeros_like(vec)
        return unit_vector(vec)

    def _novelty_score(self, z_candidate, k=5):
        if len(self.archive) == 0:
            return 0.0
        A = np.stack(self.archive, axis=0)
        diff = A - z_candidate[None, :]
        dist = np.linalg.norm(diff, axis=1)
        k = min(k, len(dist))
        idx = np.argpartition(dist, k - 1)[:k]
        return float(np.mean(dist[idx]))

    def _spacing_penalty(self, z_candidate, z_agents, i):
        diff = z_candidate[None, :] - z_agents
        dist = np.linalg.norm(diff, axis=1)
        dist[i] = np.inf
        penalty = np.maximum(0.0, self.repel_radius - dist)
        return float(np.sum(penalty))

    def step(self, z_agents):
        N, d = z_agents.shape
        assert d == self.d

        z_center = self._latent_center(z_agents)
        directions = np.zeros_like(z_agents)

        for i in range(N):
            zi = z_agents[i]
            to_center = unit_vector(z_center - zi)
            to_home = unit_vector(self.z_home[i] - zi)
            mem = unit_vector(self.memory[i]) if np.linalg.norm(self.memory[i]) > 0 else np.zeros(self.d)
            repel = self._repulsion_direction(z_agents, i)

            base = (
                self.w_c * to_center +
                self.w_h * to_home +
                self.w_m * mem +
                self.w_r * repel
            )
            if np.linalg.norm(base) > 0:
                base = unit_vector(base)
            else:
                base = np.zeros(self.d)

            best_score = -np.inf
            best_dir = np.zeros(self.d)
            best_z_cand = zi.copy()

            for _ in range(self.K):
                noise = self.rng.normal(size=self.d)
                noise = unit_vector(noise)
                canddir = unit_vector(0.7 * base + 0.3 * noise)
                z_cand = zi + self.step_size * canddir

                nov = self._novelty_score(z_cand, k=5)
                space = self._spacing_penalty(z_cand, z_agents, i)
                center_align = float(np.dot(canddir, to_center))

                score = (
                    self.lambda_nov * nov
                    - self.lambda_space * space
                    + self.lambda_center * center_align
                )

                if score > best_score:
                    best_score = score
                    best_dir = canddir
                    best_z_cand = z_cand

            new_dir = unit_vector(self.gamma * best_dir + (1.0 - self.gamma) * mem)
            directions[i] = new_dir
            self.memory[i] = new_dir
            self.archive.append(best_z_cand.copy())

        return directions


# ---------- Simple 2D GridWorld for visualization ----------

class GridWorld2D:
    """
    Continuous 2D box with simple projection to integer cells for coverage.
    """
    def __init__(self, xmin=0.0, xmax=10.0, ymin=0.0, ymax=10.0, cell_size=1.0):
        self.xmin = xmin
        self.xmax = xmax
        self.ymin = ymin
        self.ymax = ymax
        self.cell_size = cell_size

    def project(self, coords):
        """
        Project continuous coords to box and integer grid cells.
        coords: (N, 2)
        Returns:
          coords_clipped: (N, 2)
          cells: (N, 2) integer indices
        """
        x = np.clip(coords[:, 0], self.xmin, self.xmax)
        y = np.clip(coords[:, 1], self.ymin, self.ymax)
        coords_clipped = np.stack([x, y], axis=1)
        i = ((x - self.xmin) / self.cell_size).astype(int)
        j = ((y - self.ymin) / self.cell_size).astype(int)
        return coords_clipped, np.stack([i, j], axis=1)

    def grid_shape(self):
        nx = int((self.xmax - self.xmin) / self.cell_size) + 1
        ny = int((self.ymax - self.ymin) / self.cell_size) + 1
        return nx, ny


# ---------- Encoder: here latent = state (identity) ----------

def encoder(states):
    """
    For this proof-of-concept, encoder is identity: latent z = state.
    states: (N, 2)
    """
    return states.copy()


# ---------- Main experiment ----------

def run_latent_exploration_demo(
    num_agents=20,
    latent_dim=2,
    num_steps=200,
    grid_bounds=(0.0, 10.0, 0.0, 10.0),
    cell_size=0.5,
    seed=0,
):
    rng = np.random.RandomState(seed)

    # GridWorld
    xmin, xmax, ymin, ymax = grid_bounds
    env = GridWorld2D(xmin=xmin, xmax=xmax, ymin=ymin, ymax=ymax, cell_size=cell_size)

    # Latent manager: we treat latent = state, so we can bootstrap stats
    latent_mgr = LatentSpaceManager(latent_dim=latent_dim, init_c=2.0, rng=rng)

    # Bootstrap stats from uniform random positions in the box
    bootstrap_states = np.column_stack([
        rng.uniform(xmin, xmax, size=500),
        rng.uniform(ymin, ymax, size=500)
    ])
    z_bootstrap = encoder(bootstrap_states)
    latent_mgr.update_stats(z_bootstrap)

    # Initial latent positions via LHS in standardized latent
    z_init = latent_mgr.lhs_initial_latents(num_agents)  # (N, 2)

    # Project into GridWorld box to get initial states
    coords_init, _ = env.project(z_init)

    # Planner
    planner = LatentExplorationPlanner(
        latent_dim=latent_dim,
        num_candidates=16,
        center_weight=0.1,
        home_weight=0.1,
        memory_weight=0.3,
        repel_weight=0.3,
        novelty_weight=1.0,
        spacing_weight=1.0,
        center_align_weight=0.1,
        memory_smooth=0.7,
        repel_radius=1.0,
        step_size=0.5,
        rng=rng,
    )
    planner.reset_agents(encoder(coords_init))

    # Coverage tracking
    nx, ny = env.grid_shape()
    visited = np.zeros((nx, ny), dtype=np.int32)

    coords = coords_init.copy()

    for t in range(num_steps):
        # Encode current states to latent
        z_agents = encoder(coords)

        # Get latent directions
        directions = planner.step(z_agents)  # (N, 2)

        # Move in continuous space: coords_new = coords + step_size * directions
        coords_new = coords + planner.step_size * directions

        # Project back into box and update coverage
        coords, cells = env.project(coords_new)
        for (i, j) in cells:
            if 0 <= i < nx and 0 <= j < ny:
                visited[i, j] += 1

    # Plot coverage heatmap
    plt.figure(figsize=(6, 5))
    plt.imshow(visited.T, origin='lower',
               extent=[xmin, xmax, ymin, ymax],
               cmap='viridis')
    plt.colorbar(label='Visit count')
    plt.scatter(coords[:, 0], coords[:, 1], c='red', s=20, label='Final agent positions')
    plt.title('Multi-agent coverage with latent exploration planner')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.legend()
    plt.tight_layout()
    plt.show()

    return visited, coords


if __name__ == "__main__":
    visited, final_coords = run_latent_exploration_demo()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Reuse GridWorld2D and encoder from the previous cell:
# - GridWorld2D
# - encoder(states)


def run_single_agent_random_walk_demo(
    num_steps=200,
    grid_bounds=(0.0, 10.0, 0.0, 10.0),
    cell_size=0.5,
    step_size=0.5,
    seed=42,
):
    rng = np.random.RandomState(seed)

    xmin, xmax, ymin, ymax = grid_bounds
    env = GridWorld2D(xmin=xmin, xmax=xmax, ymin=ymin, ymax=ymax, cell_size=cell_size)

    nx, ny = env.grid_shape()
    visited = np.zeros((nx, ny), dtype=np.int32)

    # Random initial position in the box
    x0 = rng.uniform(xmin, xmax)
    y0 = rng.uniform(ymin, ymax)
    coords = np.array([[x0, y0]], dtype=np.float32)  # shape (1, 2)

    # Mark initial cell
    _, cells = env.project(coords)
    i0, j0 = cells[0]
    if 0 <= i0 < nx and 0 <= j0 < ny:
        visited[i0, j0] += 1

    for t in range(num_steps):
        # Simple isotropic random direction
        noise = rng.normal(size=2)
        norm = np.linalg.norm(noise) + 1e-8
        direction = noise / norm

        coords_new = coords + step_size * direction[None, :]
        coords, cells = env.project(coords_new)
        i, j = cells[0]
        if 0 <= i < nx and 0 <= j < ny:
            visited[i, j] += 1

    # Plot coverage heatmap
    plt.figure(figsize=(6, 5))
    plt.imshow(visited.T, origin='lower',
               extent=[xmin, xmax, ymin, ymax],
               cmap='plasma')
    plt.colorbar(label='Visit count')
    plt.scatter(coords[:, 0], coords[:, 1], c='white', s=30, label='Final position')
    plt.title('Single-agent random walk coverage')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.legend()
    plt.tight_layout()
    plt.show()

    return visited, coords


# Run single-agent baseline and multi-agent planner for side-by-side comparison
single_visited, single_final = run_single_agent_random_walk_demo()

multi_visited, multi_final = run_latent_exploration_demo(
    num_agents=20,
    latent_dim=2,
    num_steps=200,
    grid_bounds=(0.0, 10.0, 0.0, 10.0),
    cell_size=0.5,
    seed=0,
)

In [ ]:
def coverage_fraction(visited):
    return np.mean(visited > 0)

print("Single-agent coverage:", coverage_fraction(single_visited))
print("Multi-agent coverage:", coverage_fraction(multi_visited))